# DNAscape data management

A compact workflow for turning plain tables and genomic signal tracks into DNAscape-ready arrays.

The examples use one path variable at a time so users can replace any input with their own CSV, TSV, TXT, BigWig, or bedGraph file.

## Sections

1. Setup
2. DNAscape data conventions
3. Load DNAscape-ready files
4. Manipulate arrays
5. Save prepared data
6. Import genomic tracks
7. Download public replication-genomics tracks
8. Replication-genomics examples
9. Checklist

## 1. Setup

Import DNAscape

In [ ]:
%reload_ext autoreload
%autoreload 2

from dnascape import *

## 2. DNAscape data conventions

DNAscape uses simple shapes:

- space-only data: one value per genomic bin, shape `(n,)`;
- time-only data: one row across time, shape `(1, T)`;
- space-time data: genomic positions by time points or conditions, shape `(n, T)`;
- chromosome-aware tables: explicit `chrom`, `start_bp`, `end_bp`, `position_bp`, and signal columns.

In [ ]:
space = np.linspace(0, 1, 6)
time = np.linspace(0, 1, 4).reshape(1, -1)
spacetime = np.column_stack([space, 1 - space])

{
    "space": space.shape,
    "time": time.shape,
    "spacetime": spacetime.shape,
}

## 3. Load DNAscape-ready files

### CSV, TSV, and TXT tables

Use `rload` when the file already contains DNAscape-ready columns or numeric arrays. Chromosome-labelled tables can be loaded as tables or narrowed to one chromosome vector.

In [ ]:
USER_INPUT_PATH = "data/replication_timing/replication_timing_H1.tsv"
chromosome = "chr1"

timing_table = rload(USER_INPUT_PATH, as_table=True)
timing_chr1 = rload(USER_INPUT_PATH, chromosome=chromosome, data_form="space")

timing_table.head(), timing_chr1.shape

### Space vectors

A TXT vector is already a space-only DNAscape profile when it has one numeric value per bin.

In [ ]:
ecdna_path = "data/replication_timing/replication_timing_ecDNA_1.txt"
ecdna_profile = rload(ecdna_path, data_form="space")

ecdna_profile.shape, ecdna_profile[:5]

### Convert ready formats

Use `rdata` to convert among DNAscape-ready plain formats while preserving DNAscape shape conventions.

In [ ]:
# Example: convert one chromosome from a table to a plain vector.
# rdata(USER_INPUT_PATH, target="data/output/h1_chr1.txt", chromosome="chr1", data_form="space")

## 4. Manipulate arrays

The same profile can be clipped, interpolated, smoothed, rescaled, or sliced before mapping or simulation.

In [ ]:
raw_resolution_kb = 1
region_start_kb = 25_000
region_end_kb = 37_000

start_bin = region_start_kb // raw_resolution_kb
end_bin = region_end_kb // raw_resolution_kb

region = pd.Series(timing_chr1[start_bin:end_bin], dtype=float)
region = region.interpolate(limit_direction="both")
region_smooth = region.rolling(window=51, center=True, min_periods=1).mean()
region_scaled = (region_smooth - region_smooth.min()) / (region_smooth.max() - region_smooth.min())

ready_profile = pd.DataFrame(
    {
        "position_kb": np.arange(region_scaled.size) * raw_resolution_kb + region_start_kb,
        "replication_timing": region.to_numpy(),
        "replication_timing_smooth": region_scaled.to_numpy(),
    }
)

ready_profile.head()

## 5. Save prepared data

Use `rsave` for arrays and tables. TSV/CSV keep labels; TXT keeps the numeric vector for direct DNAscape use.

In [ ]:
USER_OUTPUT_TABLE = "data/output/timing_prepared.tsv"

rsave(ready_profile, USER_OUTPUT_TABLE, decimal_places=4)

In [ ]:
USER_OUTPUT_VECTOR = "data/output/timing_smooth.txt"

rsave(ready_profile["replication_timing_smooth"].to_numpy(), USER_OUTPUT_VECTOR, data_form="space")

## 6. Import genomic tracks

`rtrack` bins genomic coordinates into DNAscape arrays. `rdata` uses the same converter when the source is BigWig or bedGraph and can export the result directly.

### bedGraph and BDG

bedGraph-like files contain `chrom`, `start`, `end`, and `signal`. The same call works for `.bedgraph`, `.bdg`, and gzip-compressed versions.

In [ ]:
bedgraph_example = pd.DataFrame(
    {
        "chrom": ["chr1", "chr1", "chr1", "chr1"],
        "start": [0, 10_000, 30_000, 50_000],
        "end": [10_000, 20_000, 40_000, 60_000],
        "signal": [0.2, 0.6, 0.4, 0.9],
    }
)

origin_table = rtrack(
    bedgraph_example,
    source_format="bedgraph",
    chromosome="chr1",
    start=0,
    end=60_000,
    bin_size=10_000,
    aggregation="mean",
    missing="zero",
    smooth_window=3,
    normalize="minmax",
    as_table=True,
    value_name="origin_enrichment",
)

origin_array = origin_table["origin_enrichment"].to_numpy()
origin_table

In [ ]:
# Export the same binned track as a DNAscape-compatible TSV.
# rdata(
#     bedgraph_example,
#     source_format="bedgraph",
#     target="data/output/origin_enrichment.tsv",
#     chromosome="chr1",
#     start=0,
#     end=60_000,
#     bin_size=10_000,
#     missing="zero",
# )

### BigWig and BW

BigWig import uses the same options. Install the optional `pyBigWig` dependency to read `.bw` and `.bigwig` files.

In [ ]:
bigwig_path = "data/repliseq/PC3DM_S1_10000.bw"

try:
    repliseq_bw = rtrack(
        bigwig_path,
        chromosome="chr1",
        start=0,
        end=5_000_000,
        bin_size=10_000,
        aggregation="mean",
        interpolation="linear",
        smooth_window=5,
        normalize="minmax",
        as_table=True,
        value_name="repliseq_timing",
    )
except ImportError as exc:
    repliseq_bw = None
    print(exc)

None if repliseq_bw is None else repliseq_bw.head()

## 7. Download public replication-genomics tracks

These examples download small public processed tracks directly from ENCODE and GEO, then convert one chromosome or contig into DNAscape-ready arrays. The main human examples use `chr1`, `0-5 Mb`, and `50 kb` bins so the notebook stays light.

Public sources used here:

- ENCODE `ENCFF995YGM`: LNCaP Repli-seq relative replication signal, processed bedGraph, hg19.
- ENCODE `ENCFF001GNR` and `ENCFF001GOA`: GM12878 Repli-seq S1 and S4 percentage-normalized BigWig signals, hg19.
- GEO `GSE205673`: HEK293T origin-enrichment-like Loess-normalized bedGraph.
- GEO `GSE210487`: Tetrahymena RFD/fork-directionality 200 bp bedGraph.

In [ ]:
from pathlib import Path
import subprocess
import urllib.request

PUBLIC_DATA_DIR = Path("data/public_tracks")
PUBLIC_OUTPUT_DIR = Path("data/output")
PUBLIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_TRACKS = {
    "encode_lnc_repliseq": {
        "label": "ENCODE LNCaP Repli-seq relative replication signal",
        "url": "https://www.encodeproject.org/files/ENCFF995YGM/@@download/ENCFF995YGM.bed.gz",
        "path": PUBLIC_DATA_DIR / "ENCFF995YGM_lnc_repliseq_relative_signal.bedgraph.gz",
        "format": "bedgraph",
    },
    "geo_origin_enrichment": {
        "label": "GEO GSE205673 HEK293T origin-enrichment-like signal",
        "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE205nnn/GSE205673/suppl/GSE205673_HEK293T-H4WT_target_qnor.Loess.bedGraph.gz",
        "path": PUBLIC_DATA_DIR / "GSE205673_HEK293T_H4WT_origin_enrichment.bedgraph.gz",
        "format": "bedgraph",
    },
    "geo_rfd": {
        "label": "GEO GSE210487 Tetrahymena RFD / fork directionality",
        "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nnn/GSE210487/suppl/GSE210487_Kapler_428-1_RFD_200bp.bedgraph.gz",
        "path": PUBLIC_DATA_DIR / "GSE210487_Kapler_428_1_RFD_200bp.bedgraph.gz",
        "format": "bedgraph",
    },
    "encode_gm12878_s1": {
        "label": "ENCODE GM12878 Repli-seq S1 percentage-normalized signal",
        "url": "https://www.encodeproject.org/files/ENCFF001GNR/@@download/ENCFF001GNR.bigWig",
        "path": PUBLIC_DATA_DIR / "ENCFF001GNR_GM12878_repliseq_S1.bigWig",
        "format": "bigwig",
    },
    "encode_gm12878_s4": {
        "label": "ENCODE GM12878 Repli-seq S4 percentage-normalized signal",
        "url": "https://www.encodeproject.org/files/ENCFF001GOA/@@download/ENCFF001GOA.bigWig",
        "path": PUBLIC_DATA_DIR / "ENCFF001GOA_GM12878_repliseq_S4.bigWig",
        "format": "bigwig",
    },
}

def download_public_track(track_key, overwrite=False):
    """Download a public example track with Python, falling back to curl."""
    track = PUBLIC_TRACKS[track_key]
    output_path = Path(track["path"])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and output_path.stat().st_size > 0 and not overwrite:
        return output_path
    try:
        urllib.request.urlretrieve(track["url"], output_path)
    except Exception:
        subprocess.run(["curl", "-L", track["url"], "-o", str(output_path)], check=True)
    return output_path

pd.DataFrame(
    [
        {"key": key, "label": value["label"], "format": value["format"], "path": str(value["path"])}
        for key, value in PUBLIC_TRACKS.items()
    ]
)

### Public Repli-seq: ENCODE bedGraph to DNAscape array

This converts the ENCODE LNCaP relative replication signal on human `chr1` into a 50 kb DNAscape space array and coordinate table.

In [ ]:
PUBLIC_CHROMOSOME = "chr1"
PUBLIC_START_BP = 0
PUBLIC_END_BP = 5_000_000
PUBLIC_BIN_SIZE_BP = 50_000

public_repliseq_path = download_public_track("encode_lnc_repliseq")

public_repliseq = rdata(
    public_repliseq_path,
    source_format="bedgraph",
    target=PUBLIC_OUTPUT_DIR / "encode_lnc_repliseq_chr1_50kb.tsv",
    chromosome=PUBLIC_CHROMOSOME,
    start=PUBLIC_START_BP,
    end=PUBLIC_END_BP,
    bin_size=PUBLIC_BIN_SIZE_BP,
    aggregation="mean",
    interpolation="linear",
    smooth_window=3,
    normalize="minmax",
    value_name="replication_timing_signal",
    decimal_places=5,
    return_data=True,
)

public_repliseq_table = public_repliseq["table"]
public_repliseq_array = public_repliseq["values"]

rsave(
    public_repliseq_array,
    PUBLIC_OUTPUT_DIR / "encode_lnc_repliseq_chr1_50kb.txt",
    data_form="space",
)

public_repliseq_table.head(), public_repliseq_array.shape

In [ ]:
public_resolution_x_kb = PUBLIC_BIN_SIZE_BP / 1_000

# This array is now ready for DNAscape workflows. Choose the rmap source key
# that matches the biological interpretation of your processed signal.
# Example skeleton:
# mapped_initiation_rate = rmap(
#     source={"replication_timing": public_repliseq_array, "fork_speed": 1.4},
#     target="initiation_rate",
#     assumptions={"resolution_x": public_resolution_x_kb, "target_form": "space"},
# )

{
    "array_shape": public_repliseq_array.shape,
    "resolution_x_kb": public_resolution_x_kb,
    "table_output": str(public_repliseq["path"]),
}

### Public origin-enrichment-like track: GEO bedGraph

This example uses a Loess-normalized bedGraph from `GSE205673` as an SNS/Ini-like origin-enrichment signal. The output is a positive, smoothed, min-max-scaled spatial profile.

In [ ]:
public_origin_path = download_public_track("geo_origin_enrichment")

public_origin_table = rtrack(
    public_origin_path,
    source_format="bedgraph",
    chromosome="chr1",
    start=0,
    end=5_000_000,
    bin_size=50_000,
    aggregation="mean",
    interpolation="linear",
    smooth_window=3,
    normalize="minmax",
    as_table=True,
    value_name="origin_enrichment",
)

public_origin_array = public_origin_table["origin_enrichment"].to_numpy()
rsave(public_origin_table, PUBLIC_OUTPUT_DIR / "geo_origin_enrichment_chr1_50kb.tsv", decimal_places=5)
rsave(public_origin_array, PUBLIC_OUTPUT_DIR / "geo_origin_enrichment_chr1_50kb.txt", data_form="space")

public_origin_table.head(), public_origin_array.shape

### Public fork-directionality-like track: GEO RFD bedGraph

RFD values are signed, so `normalize="maxabs"` keeps the directionality centered around zero. This GEO example uses Tetrahymena contig `chr_137`.

In [ ]:
public_rfd_path = download_public_track("geo_rfd")

public_rfd_table = rtrack(
    public_rfd_path,
    source_format="bedgraph",
    chromosome="chr_137",
    start=0,
    end=200_000,
    bin_size=1_000,
    aggregation="mean",
    interpolation="linear",
    smooth_window=5,
    normalize="maxabs",
    as_table=True,
    value_name="fork_directionality",
)

public_rfd_array = public_rfd_table["fork_directionality"].to_numpy()
rsave(public_rfd_table, PUBLIC_OUTPUT_DIR / "geo_rfd_chr137_1kb.tsv", decimal_places=5)
rsave(public_rfd_array, PUBLIC_OUTPUT_DIR / "geo_rfd_chr137_1kb.txt", data_form="space")

public_rfd_table.head(), public_rfd_array.shape

### Optional BigWig space-time example: GM12878 S-phase fractions

If `pyBigWig` is installed, download two ENCODE GM12878 Repli-seq BigWigs and combine S1 and S4 into a `(space, condition)` matrix. This is the same shape DNAscape uses for space-time data.

In [ ]:
try:
    gm12878_s1 = download_public_track("encode_gm12878_s1")
    gm12878_s4 = download_public_track("encode_gm12878_s4")

    gm12878_matrix = rtrack(
        [gm12878_s1, gm12878_s4],
        chromosome="chr1",
        start=0,
        end=5_000_000,
        bin_size=50_000,
        labels=["S1", "S4"],
        interpolation="linear",
        smooth_window=3,
        normalize="minmax",
        data_form="spacetime",
        return_metadata=True,
    )

    rsave(
        gm12878_matrix["table"],
        PUBLIC_OUTPUT_DIR / "encode_gm12878_repliseq_S1_S4_chr1_50kb.tsv",
        decimal_places=5,
    )
except ImportError as exc:
    gm12878_matrix = None
    print(exc)

None if gm12878_matrix is None else (gm12878_matrix["values"].shape, gm12878_matrix["table"].head())

## 8. Replication-genomics examples

### Repli-seq, BrdU-seq, and EdU-seq

These tracks are usually interpreted as replication timing, replicated fraction, or enrichment profiles. Use mean binning, interpolate sparse gaps, then scale if the downstream workflow expects a bounded signal.

In [ ]:
repliseq_ready = origin_table.rename(columns={"origin_enrichment": "replication_timing_like"})
repliseq_ready.head()

### SNS-seq and Ini-seq

Origin-enrichment tracks often become initiation-propensity profiles. Keep zeros meaningful, smooth locally, and normalize after smoothing.

In [ ]:
origin_profile = rtrack(
    bedgraph_example,
    source_format="bedgraph",
    chromosome="chr1",
    start=0,
    end=60_000,
    bin_size=10_000,
    missing="zero",
    smooth_window=3,
    normalize="minmax",
)

origin_profile

### OK-seq and FORK-seq

Directionality tracks can be signed. Use mean aggregation, preserve the sign, and normalize with `maxabs` when the useful range is symmetric around zero.

In [ ]:
fork_direction_example = bedgraph_example.copy()
fork_direction_example["signal"] = [-0.8, -0.2, 0.3, 0.9]

fork_direction = rtrack(
    fork_direction_example,
    source_format="bedgraph",
    chromosome="chr1",
    start=0,
    end=60_000,
    bin_size=10_000,
    interpolation="linear",
    smooth_window=1,
    normalize="maxabs",
    as_table=True,
    value_name="fork_directionality",
)

fork_direction

### Space-time or multi-condition tracks

Pass several aligned tracks to `rtrack` to create a matrix where rows are genomic bins and columns are time points, fractions, or conditions.

In [ ]:
early = bedgraph_example.copy()
late = bedgraph_example.copy()
late["signal"] = 1.0 - late["signal"]

timing_matrix = rtrack(
    [early, late],
    source_format="bedgraph",
    chromosome="chr1",
    start=0,
    end=60_000,
    bin_size=10_000,
    labels=["early", "late"],
    missing="zero",
    data_form="spacetime",
    return_metadata=True,
)

timing_matrix["values"].shape, timing_matrix["table"]

## 9. Checklist

Before using a profile in `rmap` or `rsim`, confirm:

- the genomic bin size matches the intended model resolution;
- chromosome and interval choices are explicit for large genomes;
- missing bins were handled intentionally;
- smoothing and normalization preserve the biological meaning;
- exported arrays follow DNAscape shapes: `(n,)`, `(1, T)`, or `(n, T)`.